**1. Setup & Imports**

In [1]:
import os
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.model_selection import KFold
from scipy.stats import ttest_rel
from PIL import Image
from collections import Counter


**2. Paths to Dataset**

In [2]:
IMG_PATH = "Lung_CT_nodule/images"
MASK_PATH = "Lung_CT_nodule/masks"
TEST_IMG_PATH = "Lung_CT_nodule/t_images"
TEST_MASK_PATH = "Lung_CT_nodule/t_masks"

**3. Dataset Alignment & Size Diagnostics**

In [3]:
import random
import tensorflow as tf

def check_dataset_alignment(image_dir, mask_dir):
    image_files = sorted([f for f in os.listdir(image_dir) if f.lower().endswith((".jpg",".jpeg",".png"))])
    mask_files  = sorted([f for f in os.listdir(mask_dir) if f.lower().endswith((".jpg",".jpeg",".png"))])

    image_names = set([os.path.splitext(f)[0] for f in image_files])
    mask_names  = set([os.path.splitext(f)[0] for f in mask_files])

    missing_masks  = image_names - mask_names
    missing_images = mask_names - image_names

    print(f"Total images: {len(image_files)}")
    print(f"Total masks : {len(mask_files)}")
    print(f"Aligned pairs: {len(image_names & mask_names)}")
    print(f"Images without masks: {len(missing_masks)}")
    print(f"Masks without images: {len(missing_images)}")

    if missing_masks:
        print("⚠️ Images without masks (first 5):", list(missing_masks)[:5])
    if missing_images:
        print("⚠️ Masks without images (first 5):", list(missing_images)[:5])


def check_size_mismatches(image_dir, mask_dir, sample=20, full_scan=False):
    image_files = sorted([f for f in os.listdir(image_dir) if f.lower().endswith((".jpg",".jpeg",".png"))])
    mask_files  = sorted([f for f in os.listdir(mask_dir) if f.lower().endswith((".jpg",".jpeg",".png"))])

    image_names = {os.path.splitext(f)[0]: f for f in image_files}
    mask_names  = {os.path.splitext(f)[0]: f for f in mask_files}
    common = sorted(set(image_names.keys()) & set(mask_names.keys()))

    if not common:
        print("No aligned pairs found.")
        return

    if full_scan:
        pairs = common
    else:
        pairs = random.sample(common, min(sample, len(common)))

    mismatches = []
    for name in pairs:
        img_path = os.path.join(image_dir, image_names[name])
        mask_path = os.path.join(mask_dir, mask_names[name])

        img = tf.io.decode_image(tf.io.read_file(img_path)).shape[:2]
        mask = tf.io.decode_image(tf.io.read_file(mask_path)).shape[:2]

        if img != mask:
            mismatches.append((name, img, mask))

    print(f"Checked {len(pairs)} pairs, mismatches: {len(mismatches)}")
    if mismatches:
        print("⚠️ Example mismatches:", mismatches[:5])


# ---- Run diagnostics ----
print("TRAIN dataset check:")
check_dataset_alignment(IMG_PATH, MASK_PATH)
check_size_mismatches(IMG_PATH, MASK_PATH, sample=20, full_scan=False)

print("\nTEST dataset check:")
check_dataset_alignment(TEST_IMG_PATH, TEST_MASK_PATH)
check_size_mismatches(TEST_IMG_PATH, TEST_MASK_PATH, sample=20, full_scan=False)


TRAIN dataset check:
Total images: 2208
Total masks : 2208
Aligned pairs: 2208
Images without masks: 0
Masks without images: 0
Checked 20 pairs, mismatches: 0

TEST dataset check:
Total images: 552
Total masks : 552
Aligned pairs: 552
Images without masks: 0
Masks without images: 0
Checked 20 pairs, mismatches: 0


# **4. Preprocessing & Augmentation**

In [4]:
IMG_SIZE = 128

def preprocess_image(image):
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    image = tf.cast(image, tf.float32) / 255.0
    return image

def preprocess_mask(mask):
    mask = tf.image.resize(mask, (IMG_SIZE, IMG_SIZE), method="nearest")
    mask = tf.cast(mask, tf.float32)
    return tf.where(mask > 0.5, 1.0, 0.0)

def augment_image_and_mask(image, mask, seed=(1, 2)):
    # convert to proper 2D int32 seeds
    seed = tf.constant(seed, dtype=tf.int32)

    # use different variations by offsetting the two elements
    seed1 = seed
    seed2 = seed + tf.constant([1, 0], dtype=tf.int32)
    seed3 = seed + tf.constant([2, 0], dtype=tf.int32)

    # flips
    image = tf.image.stateless_random_flip_left_right(image, seed1)
    mask  = tf.image.stateless_random_flip_left_right(mask, seed1)

    image = tf.image.stateless_random_flip_up_down(image, seed2)
    mask  = tf.image.stateless_random_flip_up_down(mask, seed2)

    # brightness
    image = tf.image.stateless_random_brightness(image, 0.1, seed=seed3)

    return image, mask

def load_image_mask(image_path, mask_path, seed=(1, 2)):
    image = tf.io.decode_jpeg(tf.io.read_file(image_path), channels=3)
    mask  = tf.io.decode_png(tf.io.read_file(mask_path), channels=1)

    image = preprocess_image(image)
    mask = preprocess_mask(mask)

    return augment_image_and_mask(image, mask, seed)


**Cell 5 Dataset Loader with Alignment**

In [5]:
def create_dataset(image_dir, mask_dir, batch_size=16, shuffle=True):
    image_files = sorted([f for f in os.listdir(image_dir) if f.lower().endswith((".jpg",".jpeg",".png"))])
    mask_files  = sorted([f for f in os.listdir(mask_dir) if f.lower().endswith((".jpg",".jpeg",".png"))])

    image_names = {os.path.splitext(f)[0]: f for f in image_files}
    mask_names  = {os.path.splitext(f)[0]: f for f in mask_files}
    common = sorted(set(image_names.keys()) & set(mask_names.keys()))

    aligned_images = [os.path.join(image_dir, image_names[name]) for name in common]
    aligned_masks  = [os.path.join(mask_dir, mask_names[name]) for name in common]

    print(f"Using {len(aligned_images)} aligned pairs from {image_dir} and {mask_dir}")

    ds = tf.data.Dataset.from_tensor_slices((aligned_images, aligned_masks))
    ds = ds.map(lambda x, y: load_image_mask(x, y), num_parallel_calls=tf.data.AUTOTUNE)

    if shuffle:
        ds = ds.shuffle(buffer_size=100)

    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

# Now create datasets safely
train_ds = create_dataset(IMG_PATH, MASK_PATH)
val_ds   = create_dataset(TEST_IMG_PATH, TEST_MASK_PATH, shuffle=False)


Using 2208 aligned pairs from data_wound_seg/train_images and data_wound_seg/train_masks
Using 552 aligned pairs from data_wound_seg/test_images and data_wound_seg/test_masks


**DLU Activation function**

In [6]:
from tensorflow.keras import layers, models

# --- Custom Trainable DLU ---
class DLU(layers.Layer):
    def __init__(self, beta_init=0.0, **kwargs):
        super(DLU, self).__init__(**kwargs)
        self.beta_init = beta_init

    def build(self, input_shape):
        self.beta = self.add_weight(
            name="beta",
            shape=(1,),
            initializer=tf.keras.initializers.Constant(self.beta_init),
            trainable=True,
            constraint=tf.keras.constraints.MinMaxNorm(min_value= 0, max_value=0.01)
        )
        super(DLU, self).build(input_shape)

    def call(self, inputs):
        return tf.maximum(inputs, self.beta * inputs)

    def get_config(self):
        config = super(DLU, self).get_config()
        config.update({"beta_init": self.beta_init})
        return config





**Litnet with LeakyReLU**

In [7]:
from tensorflow.keras import layers, models

def UNet_LeakyReLU(input_size=(128, 128, 3)):
    inputs = layers.Input(input_size)

    # Encoder
    conv1 = layers.Conv2D(32, 3, padding='same')(inputs)
    conv1 = layers.BatchNormalization()(conv1)
    conv1 = layers.LeakyReLU(alpha=0.1)(conv1)
    conv1 = layers.Conv2D(32, 3, padding='same')(conv1)
    conv1 = layers.BatchNormalization()(conv1)
    conv1 = layers.LeakyReLU(alpha=0.1)(conv1)
    pool1 = layers.MaxPooling2D(pool_size=(2, 2))(conv1)

    conv2 = layers.Conv2D(64, 3, padding='same')(pool1)
    conv2 = layers.BatchNormalization()(conv2)
    conv2 = layers.LeakyReLU(alpha=0.1)(conv2)
    conv2 = layers.Conv2D(64, 3, padding='same')(conv2)
    conv2 = layers.BatchNormalization()(conv2)
    conv2 = layers.LeakyReLU(alpha=0.1)(conv2)
    pool2 = layers.MaxPooling2D(pool_size=(2, 2))(conv2)

    conv3 = layers.Conv2D(128, 3, padding='same')(pool2)
    conv3 = layers.BatchNormalization()(conv3)
    conv3 = layers.LeakyReLU(alpha=0.1)(conv3)
    conv3 = layers.Conv2D(128, 3, padding='same')(conv3)
    conv3 = layers.BatchNormalization()(conv3)
    conv3 = layers.LeakyReLU(alpha=0.1)(conv3)
    pool3 = layers.MaxPooling2D(pool_size=(2, 2))(conv3)

    # Bottleneck
    conv4 = layers.Conv2D(256, 3, padding='same')(pool3)
    conv4 = layers.BatchNormalization()(conv4)
    conv4 = layers.LeakyReLU(alpha=0.1)(conv4)
    conv4 = layers.Conv2D(256, 3, padding='same')(conv4)
    conv4 = layers.BatchNormalization()(conv4)
    conv4 = layers.LeakyReLU(alpha=0.1)(conv4)
    drop4 = layers.Dropout(0.4)(conv4)

    # Decoder
    up5 = layers.Conv2DTranspose(128, 2, strides=(2, 2), padding='same')(drop4)
    merge5 = layers.concatenate([conv3, up5], axis=3)
    conv5 = layers.Conv2D(128, 3, padding='same')(merge5)
    conv5 = layers.BatchNormalization()(conv5)
    conv5 = layers.LeakyReLU(alpha=0.1)(conv5)

    up6 = layers.Conv2DTranspose(64, 2, strides=(2, 2), padding='same')(conv5)
    merge6 = layers.concatenate([conv2, up6], axis=3)
    conv6 = layers.Conv2D(64, 3, padding='same')(merge6)
    conv6 = layers.BatchNormalization()(conv6)
    conv6 = layers.LeakyReLU(alpha=0.1)(conv6)

    up7 = layers.Conv2DTranspose(32, 2, strides=(2, 2), padding='same')(conv6)
    merge7 = layers.concatenate([conv1, up7], axis=3)
    conv7 = layers.Conv2D(32, 3, padding='same')(merge7)
    conv7 = layers.BatchNormalization()(conv7)
    conv7 = layers.LeakyReLU(alpha=0.1)(conv7)

    # Output
    outputs = layers.Conv2D(1, 1, activation='sigmoid')(conv7)

    return models.Model(inputs, outputs, name="U-Net-LeakyReLU")


**Litnet with ReLU**

In [8]:
def UNet_ReLU(input_size=(128, 128, 3)):
    inputs = layers.Input(input_size)

    # Encoder
    conv1 = layers.Conv2D(32, 3, padding='same')(inputs)
    conv1 = layers.BatchNormalization()(conv1)
    conv1 = layers.ReLU()(conv1)
    conv1 = layers.Conv2D(32, 3, padding='same')(conv1)
    conv1 = layers.BatchNormalization()(conv1)
    conv1 = layers.ReLU()(conv1)
    pool1 = layers.MaxPooling2D(pool_size=(2, 2))(conv1)

    conv2 = layers.Conv2D(64, 3, padding='same')(pool1)
    conv2 = layers.BatchNormalization()(conv2)
    conv2 = layers.ReLU()(conv2)
    conv2 = layers.Conv2D(64, 3, padding='same')(conv2)
    conv2 = layers.BatchNormalization()(conv2)
    conv2 = layers.ReLU()(conv2)
    pool2 = layers.MaxPooling2D(pool_size=(2, 2))(conv2)

    conv3 = layers.Conv2D(128, 3, padding='same')(pool2)
    conv3 = layers.BatchNormalization()(conv3)
    conv3 = layers.ReLU()(conv3)
    conv3 = layers.Conv2D(128, 3, padding='same')(conv3)
    conv3 = layers.BatchNormalization()(conv3)
    conv3 = layers.ReLU()(conv3)
    pool3 = layers.MaxPooling2D(pool_size=(2, 2))(conv3)

    # Bottleneck
    conv4 = layers.Conv2D(256, 3, padding='same')(pool3)
    conv4 = layers.BatchNormalization()(conv4)
    conv4 = layers.ReLU()(conv4)
    conv4 = layers.Conv2D(256, 3, padding='same')(conv4)
    conv4 = layers.BatchNormalization()(conv4)
    conv4 = layers.ReLU()(conv4)
    drop4 = layers.Dropout(0.4)(conv4)

    # Decoder
    up5 = layers.Conv2DTranspose(128, 2, strides=(2, 2), padding='same')(drop4)
    merge5 = layers.concatenate([conv3, up5], axis=3)
    conv5 = layers.Conv2D(128, 3, padding='same')(merge5)
    conv5 = layers.BatchNormalization()(conv5)
    conv5 = layers.ReLU()(conv5)

    up6 = layers.Conv2DTranspose(64, 2, strides=(2, 2), padding='same')(conv5)
    merge6 = layers.concatenate([conv2, up6], axis=3)
    conv6 = layers.Conv2D(64, 3, padding='same')(merge6)
    conv6 = layers.BatchNormalization()(conv6)
    conv6 = layers.ReLU()(conv6)

    up7 = layers.Conv2DTranspose(32, 2, strides=(2, 2), padding='same')(conv6)
    merge7 = layers.concatenate([conv1, up7], axis=3)
    conv7 = layers.Conv2D(32, 3, padding='same')(merge7)
    conv7 = layers.BatchNormalization()(conv7)
    conv7 = layers.ReLU()(conv7)

    # Output
    outputs = layers.Conv2D(1, 1, activation='sigmoid')(conv7)

    return models.Model(inputs, outputs, name="U-Net-ReLU")


**Litnet with DLU**

In [9]:
def LitUNet_DLU(input_size=(128, 128, 3)):
    inputs = layers.Input(input_size)

    # Encoder
    conv1 = layers.Conv2D(32, 3, padding='same')(inputs)
    conv1 = layers.BatchNormalization()(conv1)
    conv1 = DLU()(conv1)
    conv1 = layers.Conv2D(32, 3, padding='same')(conv1)
    conv1 = layers.BatchNormalization()(conv1)
    conv1 = DLU()(conv1)
    pool1 = layers.MaxPooling2D(pool_size=(2, 2))(conv1)

    conv2 = layers.Conv2D(64, 3, padding='same')(pool1)
    conv2 = layers.BatchNormalization()(conv2)
    conv2 = DLU()(conv2)
    conv2 = layers.Conv2D(64, 3, padding='same')(conv2)
    conv2 = layers.BatchNormalization()(conv2)
    conv2 = DLU()(conv2)
    pool2 = layers.MaxPooling2D(pool_size=(2, 2))(conv2)

    conv3 = layers.Conv2D(128, 3, padding='same')(pool2)
    conv3 = layers.BatchNormalization()(conv3)
    conv3 = DLU()(conv3)
    conv3 = layers.Conv2D(128, 3, padding='same')(conv3)
    conv3 = layers.BatchNormalization()(conv3)
    conv3 = DLU()(conv3)
    pool3 = layers.MaxPooling2D(pool_size=(2, 2))(conv3)

    # Bottleneck
    conv4 = layers.Conv2D(256, 3, padding='same')(pool3)
    conv4 = layers.BatchNormalization()(conv4)
    conv4 = DLU()(conv4)
    conv4 = layers.Conv2D(256, 3, padding='same')(conv4)
    conv4 = layers.BatchNormalization()(conv4)
    conv4 = DLU()(conv4)
    drop4 = layers.Dropout(0.4)(conv4)

    # Decoder
    up5 = layers.Conv2DTranspose(128, 2, strides=(2, 2), padding='same')(drop4)
    merge5 = layers.concatenate([conv3, up5], axis=3)
    conv5 = layers.Conv2D(128, 3, padding='same')(merge5)
    conv5 = layers.BatchNormalization()(conv5)
    conv5 = DLU()(conv5)

    up6 = layers.Conv2DTranspose(64, 2, strides=(2, 2), padding='same')(conv5)
    merge6 = layers.concatenate([conv2, up6], axis=3)
    conv6 = layers.Conv2D(64, 3, padding='same')(merge6)
    conv6 = layers.BatchNormalization()(conv6)
    conv6 = DLU()(conv6)

    up7 = layers.Conv2DTranspose(32, 2, strides=(2, 2), padding='same')(conv6)
    merge7 = layers.concatenate([conv1, up7], axis=3)
    conv7 = layers.Conv2D(32, 3, padding='same')(merge7)
    conv7 = layers.BatchNormalization()(conv7)
    conv7 = DLU()(conv7)

    # Output
    outputs = layers.Conv2D(1, 1, activation='sigmoid')(conv7)

    return models.Model(inputs, outputs, name="LitUNet-DLU")


# Runtime Benchmark (CPU vs GPU)

In [14]:
### Cell 13 – Runtime Benchmark (CPU vs GPU)
import time
import pandas as pd

# --- Benchmark Training ---
def timed_training(build_fn, name, device="/GPU:0", epochs=3):
    with tf.device(device):
        model = build_fn()
        model.compile(optimizer=tf.keras.optimizers.Adam(1e-4),
                     loss=combined_loss,
                     metrics=["accuracy", dice_coefficient, tf.keras.metrics.MeanIoU(num_classes=2)])
        start = time.time()
        model.fit(train_ds, validation_data=val_ds, epochs=epochs, verbose=0)
        end = time.time()
        avg_time = (end - start) / epochs
        print(f"{name} on {device}: {avg_time:.2f} sec/epoch")
        return avg_time

# --- Benchmark Inference ---
def measure_inference(model, dataset, device="/GPU:0", num_batches=10):
    times = []
    with tf.device(device):
        for i, (images, _) in enumerate(dataset.take(num_batches)):
            start = time.time()
            _ = model.predict(images, verbose=0)
            times.append(time.time() - start)
    avg_time = np.mean(times)
    print(f"Average inference on {device}: {avg_time:.4f} sec/batch")
    return avg_time

# --- Run Benchmarks for All Models ---
print("\n⏱️ Running Runtime Benchmarks...")
results = []
for build_fn, name in [
    (UNet_ReLU, "ReLU"),
    (UNet_LeakyReLU, "LeakyReLU"),
    (LitUNet_DLU, "DLU")
]:
    # Training speed
    gpu_train = timed_training(build_fn, name, device="/GPU:0", epochs=3)
    cpu_train = timed_training(build_fn, name, device="/CPU:0", epochs=3)

    # Inference speed (use small warm-up fit first)
    model = build_fn()
    model.compile(optimizer="adam", loss=combined_loss, metrics=[dice_coefficient])
    model.fit(train_ds.take(1), epochs=1, verbose=0)  # warm-up

    gpu_inf = measure_inference(model, val_ds, device="/GPU:0", num_batches=10)
    cpu_inf = measure_inference(model, val_ds, device="/CPU:0", num_batches=10)

    results.append({
        "Model": name,
        "GPU Train (sec/epoch)": round(gpu_train, 2),
        "CPU Train (sec/epoch)": round(cpu_train, 2),
        "GPU Inference (sec/batch)": round(gpu_inf, 4),
        "CPU Inference (sec/batch)": round(cpu_inf, 4),
        "Speedup (Train)": f"{cpu_train/gpu_train:.1f}x",
        "Speedup (Inference)": f"{cpu_inf/gpu_inf:.1f}x"
    })

# --- Summary Table ---
df_results = pd.DataFrame(results)
print("\n" + "="*70)
print("RUNTIME BENCHMARK RESULTS")
print("="*70)
print(df_results.to_string(index=False))


⏱️ Running Runtime Benchmarks...
ReLU on /GPU:0: 17.74 sec/epoch
ReLU on /CPU:0: 574.68 sec/epoch
Average inference on /GPU:0: 0.2228 sec/batch
Average inference on /CPU:0: 1.0667 sec/batch
LeakyReLU on /GPU:0: 18.96 sec/epoch
LeakyReLU on /CPU:0: 576.65 sec/epoch
Average inference on /GPU:0: 0.2329 sec/batch
Average inference on /CPU:0: 1.0728 sec/batch
DLU on /GPU:0: 20.05 sec/epoch
DLU on /CPU:0: 750.94 sec/epoch
Average inference on /GPU:0: 0.2461 sec/batch
Average inference on /CPU:0: 1.0817 sec/batch

RUNTIME BENCHMARK RESULTS
    Model  GPU Train (sec/epoch)  CPU Train (sec/epoch)  GPU Inference (sec/batch)  CPU Inference (sec/batch) Speedup (Train) Speedup (Inference)
     ReLU                  17.74                 574.68                     0.2228                     1.0667           32.4x                4.8x
LeakyReLU                  18.96                 576.65                     0.2329                     1.0728           30.4x                4.6x
      DLU             